# 08. 演習解答(付録)

本書(PDE Book)の各章末 **Exercises** の解答集です。考え方(数式・理由)を述べ、
可能なものは `pde_book` の関数で数値的に検証します。**まず自力で解いてから**読むことを勧めます。

| 対象 | 章 |
|---|---|
| 00 | 微分積分の基礎(両書共通) |
| 02–07 | PDE 各章 |

## 00. 微分積分の基礎

In [1]:
from pde_book import calculus
import sympy as sp

**00-1 割線→接線**: $f(x)=x^2$ の差の商は $\dfrac{(2+h)^2-2^2}{h}=4+h \to 4$。$h\to0$ で接線の傾き $f'(2)=4$。

In [2]:
for h in (1.0, 0.5, 0.1, 0.01):
    print(f"h={h:5.2f}  secant slope = {calculus.secant_slope(lambda x: x**2, 2.0, h):.4f}  (= 4 + h)")

h= 1.00  secant slope = 5.0000  (= 4 + h)
h= 0.50  secant slope = 4.5000  (= 4 + h)
h= 0.10  secant slope = 4.1000  (= 4 + h)
h= 0.01  secant slope = 4.0100  (= 4 + h)


**00-2 Riemann 和の規則**: midpoint は誤差 $O(\Delta x^2)$、left/right は $O(\Delta x)$。よって midpoint が最速で $1/3$ に収束。

In [3]:
f = lambda x: x**2
for n in (10, 100):
    row = {r: calculus.riemann_sum(f, 0, 1, n, r) for r in ("left", "mid", "right")}
    print(f"n={n:3d}  left={row['left']:.5f}  mid={row['mid']:.5f}  right={row['right']:.5f}  (exact 1/3={1/3:.5f})")

n= 10  left=0.28500  mid=0.33250  right=0.38500  (exact 1/3=0.33333)
n=100  left=0.32835  mid=0.33333  right=0.33835  (exact 1/3=0.33333)


**00-3 FTC**: $\int_0^x e^{-t}\,dt = 1-e^{-x}$。累積積分が解析解に一致する。

In [4]:
import numpy as np
xs = np.linspace(0, 3, 400)
F = calculus.cumulative_integral(lambda t: np.exp(-t), xs)
print("max |F - (1 - e^-x)| =", float(np.max(np.abs(F - (1 - np.exp(-xs))))))

max |F - (1 - e^-x)| = 4.4764733087010455e-06


**00-4 勾配**: $\nabla(\sin x\cos y)=(\cos x\cos y,\,-\sin x\sin y)$。$(\pi/4,\pi/4)$ では $(\tfrac12,-\tfrac12)$。

In [5]:
import numpy as np
f = lambda p: np.sin(p[0]) * np.cos(p[1])
g = calculus.gradient(f, [np.pi / 4, np.pi / 4])
print("numerical grad =", g, " | analytic = (0.5, -0.5)")

numerical grad = [ 0.5 -0.5]  | analytic = (0.5, -0.5)


**00-5 Taylor**: $\ln(1+x)=x-\tfrac{x^2}{2}+\tfrac{x^3}{3}-\tfrac{x^4}{4}+\tfrac{x^5}{5}-\cdots$。$x=0.5$ で 5 次近似の誤差を評価。

In [6]:
import numpy as np
x = sp.symbols("x")
poly = calculus.taylor_series(sp.log(1 + x), x, 0, 5)
print("Taylor(ln(1+x), order 5):", poly)
approx = calculus.taylor_approx(sp.log(1 + x), x, 0, 5)
print("error at x=0.5:", abs(float(approx(0.5)) - np.log(1.5)))

Taylor(ln(1+x), order 5): x**5/5 - x**4/4 + x**3/3 - x**2/2 + x
error at x=0.5: 0.001826558558502278


In [7]:
# Shared setup. Make the book package importable whether or not it is pip-installed,
# then fix the random seed and tidy NumPy printing.
import sys
from pathlib import Path

try:
    import pde_book  # noqa: F401
except ModuleNotFoundError:
    for _base in (Path.cwd(), *Path.cwd().parents):
        if (_base / "src" / "pde_book").is_dir():
            sys.path.insert(0, str(_base / "src"))
            break

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

In [8]:
from pde_book import grids, solvers, datasets
import numpy as np

## 02. 移流・熱・波動

**02-1**: $c<0$ では情報が右から来るので upwind は **右隣** $u_{i+1}$ を使う。波形は左へ動く。

In [9]:
g = grids.Grid1D(0, 1, 201)
x, dx = g.x, g.dx
c = -1.0
U = solvers.solve_transport(datasets.gaussian(x, 0.7, 0.05), c, dx, 0.8 * dx / abs(c), 100, "upwind")
print("peak moved from 0.70 to", round(float(x[np.argmax(U[-1])]), 3), "(leftward, since c<0)")

peak moved from 0.70 to 0.3 (leftward, since c<0)


**02-2**: 熱方程式はガウス分布を **広がるガウス分布** に保つ(分散 $\sigma^2(t)=\sigma_0^2+2\alpha t$)。標準偏差が増える。

In [10]:
g = grids.Grid1D(0, 1, 401)
x, dx = g.x, g.dx
u0 = datasets.gaussian(x, 0.5, 0.04, height=1.0)
U = solvers.solve_heat_explicit(u0, 1.0, dx, 0.4 * dx**2, 800)
def width(u):  # std of the (normalized) profile
    w = u / u.sum()
    return np.sqrt(np.sum(w * (x - np.sum(w * x)) ** 2))
print("std grows:", round(width(U[0]), 4), "->", round(width(U[-1]), 4), "(diffusion widens the Gaussian)")

std grows: 0.04 -> 0.0748 (diffusion widens the Gaussian)


**02-3**: 「はじく」(初期変位 $u_0\ne0,\,v_0=0$)は左右に分裂して伝播。「たたく」($u_0=0,\,v_0\ne0$)は速度から変位が立ち上がる。

In [11]:
g = grids.Grid1D(0, 1, 401)
x, dx = g.x, g.dx
c = 1.0
dt = 0.8 * dx / c
pluck = solvers.solve_wave(datasets.gaussian(x, 0.5, 0.04), np.zeros_like(x), c, dx, dt, 200)
hammer = solvers.solve_wave(np.zeros_like(x), datasets.gaussian(x, 0.5, 0.04), c, dx, dt, 200)
print("pluck max displacement:", round(float(np.abs(pluck).max()), 3),
      "| hammer builds displacement from velocity:", round(float(np.abs(hammer[-1]).max()), 3))

pluck max displacement: 1.0 | hammer builds displacement from velocity: 0.05


## 03. Laplace・Poisson

**03-1**: 4 辺を異なる温度に固定して定常分布を解く(各辺の値が内部へ滑らかに補間される)。

In [12]:
g = grids.Grid2D(0, 1, 0, 1, 41, 41)
b = np.zeros((g.ny, g.nx))
b[-1, :], b[0, :], b[:, 0], b[:, -1] = 1.0, 0.0, 0.5, 0.25  # top, bottom, left, right
u = solvers.solve_poisson_2d(np.zeros_like(b), g, b)
print("interior temperature stays within the boundary range:",
      round(float(u[1:-1, 1:-1].min()), 3), "..", round(float(u[1:-1, 1:-1].max()), 3))

interior temperature stays within the boundary range: 0.023 .. 0.966


**03-2**: Poisson は線形なので源 $f\to-f$ で解 $u\to-u$。湧き出しが吸い込みになり、場は **凹む**。

In [13]:
g = grids.Grid2D(0, 1, 0, 1, 51, 51)
X, Y = g.meshgrid()
src = 50.0 * np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / 0.005)
up = solvers.solve_poisson_2d(-src, g, np.zeros_like(X))
dn = solvers.solve_poisson_2d(+src, g, np.zeros_like(X))
print("center value: +source ->", round(float(up[25, 25]), 3), " | -source ->", round(float(dn[25, 25]), 3), "(sign flips)")

center value: +source -> 0.291  | -source -> -0.291 (sign flips)


**03-3**: 線形 $u=ax+by+c$ は $\nabla^2u=0$(調和)。境界に線形値を与えると内部もその線形関数(5 点ステンシルは線形に厳密)。

In [14]:
g = grids.Grid2D(0, 1, 0, 1, 21, 21)
X, Y = g.meshgrid()
u_lin = 2 * X + 3 * Y + 1
u = solvers.solve_poisson_2d(np.zeros_like(X), g, u_lin)
print("max |u - (2x+3y+1)| =", float(np.max(np.abs(u - u_lin))), "(~0: linear is harmonic)")

max |u - (2x+3y+1)| = 6.217248937900877e-15 (~0: linear is harmonic)


## 04. Fourier 級数と変換

**04-1**: 三角波の係数は $\propto 1/k^2$(矩形波は $1/k$)。$1/k^2$ は速く減衰し、跳びが無いので **Gibbs が出ず速く収束**。

In [15]:
x = np.linspace(0, 2 * np.pi, 1000)
tri = 2 / np.pi * np.arcsin(np.sin(x))  # triangle wave in [-1,1]
def tri_partial(n):
    s = np.zeros_like(x)
    for j in range(n):
        k = 2 * j + 1
        s += (-1) ** j / k**2 * np.sin(k * x)
    return 8 / np.pi**2 * s
for n in (1, 3, 10):
    print(f"triangle, {n:2d} terms: max error = {np.max(np.abs(tri_partial(n) - tri)):.4f}")

triangle,  1 terms: max error = 0.1884
triangle,  3 terms: max error = 0.0659
triangle, 10 terms: max error = 0.0193


**04-2**: Nyquist($f_s/2$)を超える正弦は **折り返し(エイリアシング)**。$f_s=10$ で $f=8$ を標本化すると $|8-10|=2$ Hz に化ける。

In [16]:
fs = 10.0
n = np.arange(0, 4, 1 / fs)
hi = np.sin(2 * np.pi * 8 * n)
spec = np.abs(np.fft.rfft(hi)) / (len(n) / 2)
freq = np.fft.rfftfreq(len(n), 1 / fs)
print("8 Hz sampled at fs=10 appears at", round(float(freq[np.argmax(spec)]), 2), "Hz (alias = |8-10| = 2)")

8 Hz sampled at fs=10 appears at 2.0 Hz (alias = |8-10| = 2)


**04-3**: 熱方程式はモード $\sin(n\pi x)$ を固有値 $(n\pi)^2$ に応じて $e^{-\alpha(n\pi)^2 t}$ で減衰させる。
比 $a_1/a_4 \propto e^{-\alpha(1^2-4^2)\pi^2 t}=e^{15\alpha\pi^2 t}$ で **増大**(高次が速く消える)。

In [17]:
g = grids.Grid1D(0, 1, 201)
x, dx = g.x, g.dx
alpha = 1.0
u0 = datasets.sine_combo(x, (1, 4), (1.0, 1.0), L=1.0)
proj = lambda u, k: 2 * np.trapezoid(u * np.sin(k * np.pi * x), x)  # mode amplitude
U = solvers.solve_heat_explicit(u0, alpha, dx, 0.4 * dx**2, 400)
t_end = 400 * 0.4 * dx**2
r_num = proj(U[-1], 1) / proj(U[-1], 4)
r_th = (proj(u0, 1) / proj(u0, 4)) * np.exp(-alpha * (1 - 16) * np.pi**2 * t_end)
print("amplitude ratio a1/a4: numerical =", round(float(r_num), 2), " theory =", round(float(r_th), 2))

amplitude ratio a1/a4: numerical = 1.81  theory = 1.81


## 05. 変数分離法

**05-1**: Neumann($u_x(0)=u_x(L)=0$)では $X''=-\lambda X$ の解で端の傾きが 0 になるのは $X_n=\cos(n\pi x/L)$。
($X_n'=-\tfrac{n\pi}{L}\sin(n\pi x/L)$ は $x=0,L$ で 0。)

In [18]:
g = grids.Grid1D(0, 1, 201)
x = g.x
for n in (1, 2, 3):
    Xp = -n * np.pi * np.sin(n * np.pi * x)  # derivative of cos(n pi x)
    print(f"cos({n} pi x): |X'(0)|={abs(Xp[0]):.2e}, |X'(L)|={abs(Xp[-1]):.2e} (Neumann satisfied)")

cos(1 pi x): |X'(0)|=0.00e+00, |X'(L)|=3.85e-16 (Neumann satisfied)
cos(2 pi x): |X'(0)|=0.00e+00, |X'(L)|=1.54e-15 (Neumann satisfied)
cos(3 pi x): |X'(0)|=0.00e+00, |X'(L)|=3.46e-15 (Neumann satisfied)


**05-2**: 3 モード混合の初期条件を熱方程式で時間発展させると、$e^{-\alpha k^2 t}$ により高次が先に消え、**最低次モードだけが残る**。

In [19]:
g = grids.Grid1D(0, 1, 201)
x, dx = g.x, g.dx
U = solvers.solve_heat_explicit(datasets.sine_combo(x, (1, 3, 6), (1.0, 0.8, 0.6)), 1.0, dx, 0.4 * dx**2, 2000)
proj = lambda u, k: 2 * np.trapezoid(u * np.sin(k * np.pi * x), x)
amps = {k: round(float(abs(proj(U[-1], k))), 4) for k in (1, 3, 6)}
print("late-time mode amplitudes:", amps, "-> mode 1 dominates")

late-time mode amplitudes: {1: 0.8209, 3: 0.1353, 6: 0.0005} -> mode 1 dominates


**05-3**: 弦の固有振動数は $f_n=n c/(2L)$、基本($n=1$)は角振動数 $\pi c/L$。$L$ を小さくすると振動数が上がる(高い音)。

In [20]:
c = 1.0
for L in (1.0, 0.5):
    print(f"L={L}: fundamental angular frequency = pi c / L = {np.pi * c / L:.3f}")
print("-> shorter string (smaller L) gives a higher pitch")

L=1.0: fundamental angular frequency = pi c / L = 3.142
L=0.5: fundamental angular frequency = pi c / L = 6.283
-> shorter string (smaller L) gives a higher pitch


## 06. 有限差分法

**06-1**: モード $\sin(kx)$ の FTCS 増幅率は $g=1-4r\sin^2(k\Delta x/2)$。$|g|\le1$ は最悪 $\sin^2=1$ で $1-4r\ge-1 \Rightarrow r\le1/2$。

In [21]:
r = 0.5
worst = 1 - 4 * r * 1.0  # sin^2 = 1
print("at r=1/2 worst amplification =", worst, "(= -1, the stability edge)")
print("r=0.6 worst amplification =", 1 - 4 * 0.6, "(< -1 -> grows, unstable)")

at r=1/2 worst amplification = -1.0 (= -1, the stability edge)
r=0.6 worst amplification = -1.4 (< -1 -> grows, unstable)


**06-2**: $C=1$ の upwind は $u_i^{n+1}=u_i-1\cdot(u_i-u_{i-1})=u_{i-1}$、すなわち **1 格子ぴったりの平行移動**=数値拡散ゼロの厳密移流。

In [22]:
g = grids.Grid1D(0, 1, 201)
x, dx = g.x, g.dx
u0 = datasets.gaussian(x, 0.3, 0.05)
U = solvers.solve_transport(u0, 1.0, dx, 1.0 * dx, 50, "upwind")  # CFL = 1 exactly
shifted = np.roll(u0, 50)  # exact shift by 50 cells
print("max |upwind(C=1) - exact shift| =", float(np.max(np.abs(U[-1] - shifted))), "(~0: no numerical diffusion)")

max |upwind(C=1) - exact shift| = 3.1554436208840472e-30 (~0: no numerical diffusion)


**06-3**: 陰解法 $(I-rL)u^{n+1}=u^n$ の行列は **三重対角**。`scipy` の `splu` で一度だけ LU 分解すれば毎ステップ前進・後退代入で速い。

In [23]:
import scipy.sparse as sp
n, r = 8, 0.5
A = sp.diags([-r * np.ones(n - 1), (1 + 2 * r) * np.ones(n), -r * np.ones(n - 1)], [-1, 0, 1]).toarray()
band = np.count_nonzero(A) - np.count_nonzero(np.diag(A)) - np.count_nonzero(np.diag(A, 1)) - np.count_nonzero(np.diag(A, -1))
print("nonzeros off the three central diagonals:", band, "(0 => tridiagonal)")

nonzeros off the three central diagonals: 0 (0 => tridiagonal)


## 07. 応用

**07-1**: 2 次元 FTCS の増幅率は $g=1-4r(\sin^2\tfrac{k_x\Delta x}{2}+\sin^2\tfrac{k_y\Delta y}{2})$、最悪 $1-8r\ge-1 \Rightarrow r\le1/4$。
(1 次元 $1/2$ の半分。隣接点が 4 方向に増えるため。)

In [24]:
print("2-D worst amplification at r=1/4:", 1 - 8 * 0.25, "(= -1, the 2-D stability edge)")

2-D worst amplification at r=1/4: -1.0 (= -1, the 2-D stability edge)


**07-2**: Black-Scholes 価格は満期で折れ線(payoff)に近づき二階微分が鋭くなるので、$t\to T$ で有限差分の **残差(誤差)が増大**。

In [25]:
from scipy.stats import norm
K, rr, sig, T = 1.0, 0.05, 0.2, 1.0
def bs(S, t):
    tau = max(T - t, 1e-9)
    d1 = (np.log(S / K) + (rr + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))
    return S * norm.cdf(d1) - K * np.exp(-rr * tau) * norm.cdf(d1 - sig * np.sqrt(tau))
S0, h, dtau = 1.0, 1e-4, 1e-5
for t0 in (0.4, 0.9, 0.99):
    Vt = (bs(S0, t0 + dtau) - bs(S0, t0 - dtau)) / (2 * dtau)
    Vss = (bs(S0 + h, t0) - 2 * bs(S0, t0) + bs(S0 - h, t0)) / h**2
    res = Vt + 0.5 * sig**2 * S0**2 * Vss + rr * S0 * (bs(S0 + h, t0) - bs(S0 - h, t0)) / (2 * h) - rr * bs(S0, t0)
    print(f"t={t0:.2f} (T-t={T - t0:.2f}): PDE residual = {res:.2e}")

t=0.40 (T-t=0.60): PDE residual = -2.02e-09
t=0.90 (T-t=0.10): PDE residual = -2.70e-08


t=0.99 (T-t=0.01): PDE residual = -8.81e-07


**07-3**: 画像拡散はステップが進むほどエッジが鈍る。勾配の大きさ(エッジの強さ)の総和が単調に減少する。

In [26]:
img = datasets.make_test_image(96, seed=0)
u = img.copy()
def edge_strength(a):
    gx = np.diff(a, axis=1); gy = np.diff(a, axis=0)
    return float(np.abs(gx).sum() + np.abs(gy).sum())
print(f"step   0: edge strength = {edge_strength(u):.0f}")
for k in range(1, 41):
    lap = np.roll(u, 1, 0) + np.roll(u, -1, 0) + np.roll(u, 1, 1) + np.roll(u, -1, 1) - 4 * u
    u = u + 0.2 * lap
    if k in (10, 40):
        print(f"step {k:3d}: edge strength = {edge_strength(u):.0f}  (edges blur away)")

step   0: edge strength = 2481
step  10: edge strength = 220  (edges blur away)
step  40: edge strength = 160  (edges blur away)
